# OPCUA Mock Server

This is an outline for an OPC UA Server for our fischertechnik use case

## Dependencies
I do not know how you guys manage your dependencies, this is just my preference of making sure the dependencies are installed ;)

In [1]:
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [2]:
packages = ['scipy', 'numpy', 'asyncua']

In [3]:
id = 0
for pack in packages:
    print("("+str(id) + "/" + str(len(packages))+") "+"installing "+ pack + ".")
    install(pack)
    id += 1
    print("("+str(id)+ "/"+ str(len(packages))+ ") "+ pack + " installed.")

(0/3) installing scipy.
(1/3) scipy installed.
(1/3) installing numpy.
(2/3) numpy installed.
(2/3) installing asyncua.
(3/3) asyncua installed.


## Settings

Code for some reason does not run in notebook --> Annotations?/Decorators?

In [ ]:
import asyncio
import copy
import logging
from datetime import datetime
import time
from math import sin


from asyncua import ua, uamethod, Server


_logger = logging.getLogger(__name__)


class SubHandler(object):

    """
    Subscription Handler. To receive events from server for a subscription
    """

    def datachange_notification(self, node, val, data):
        _logger.warning("Python: New data change event %s %s", node, val)

    def event_notification(self, event):
        _logger.warning("Python: New event %s", event)


# method to be exposed through server
def func(parent, variant):
    ret = False
    if variant.Value % 2 == 0:
        ret = True
    return [ua.Variant(ret, ua.VariantType.Boolean)]


# method to be exposed through server
# uses a decorator to automatically convert to and from variants
@uamethod
def multiply(parent, x, y):
    _logger.warning("multiply method call with parameters: %s %s", x, y)
    return x * y


async def main():

    server = Server()
    await server.init()
    # server.disable_clock()  #for debuging
    server.set_endpoint("opc.tcp://localhost:4840/fischertechnikSample")
    server.set_server_name("Sample Server for fischertechnik")
    # set all possible endpoint policies for clients to connect through
    server.set_security_policy(
        [
            ua.SecurityPolicyType.NoSecurity,
            ua.SecurityPolicyType.Basic256Sha256_SignAndEncrypt,
            ua.SecurityPolicyType.Basic256Sha256_Sign,
        ]
    )

    # setup our own namespace
    uri = "http://digtwin.mbdo.eu/fischertech"
    idx = await server.register_namespace(uri)

    # create a new node type we can instantiate in our address space
    dev = await server.nodes.base_object_type.add_object_type(idx, "Conveyor")
    await (await dev.add_variable(idx, "sensor1", 1.0)).set_modelling_rule(True)
    await (await dev.add_property(idx, "device_id", "0340")).set_modelling_rule(True)
    ctrl = await dev.add_object(idx, "controller")
    await ctrl.set_modelling_rule(True)
    await (await ctrl.add_property(idx, "state", "Idle")).set_modelling_rule(True)

    # populating our address space

    # First a folder to organise our nodes
    folder = await server.nodes.objects.add_folder(idx, "devices")
    # instanciate one instance of our device
    device1 = await server.nodes.objects.add_object(idx, "Device0001", dev)
    device1_var1 = await device1.get_child(
        [f"{idx}:controller", f"{idx}:state"]
    )  # get proxy to our device state variable
    # create directly some objects and variables
    server_object = await server.nodes.objects.add_object(idx, "Conveyor")
    server_var = await server_object.add_variable(idx, "EntryLight", 6.7)
    await server_var.set_writable()  # Set EntryLight to be writable by clients
    string_var1 = await server_object.add_variable(idx, "ProductOnConveyor", "Product is on conveyor")
    await string_var1.set_writable()  # Set MyVariable to be writable by clients
    datetime_var = await server_object.add_variable(idx, "Time", datetime.utcnow())
    await datetime_var.set_writable()  # Set MyVariable to be writable by clients
    array_var = await server_object.add_variable(idx, "processing", [0.0, 1.0])
    myuintvar = await server_object.add_variable(idx, "myuintvar", ua.UInt16(4))
    # if we need strong typing, properties or functions these following lines become useful
    #await myobj.add_variable(idx, "myStronglytTypedVariable", ua.Variant([], ua.VariantType.UInt32))
    #await myarrayvar.set_writable(True)
    #myprop = await myobj.add_property(idx, "myproperty", "I am a property")
    #mymethod = await myobj.add_method(
    #    idx, "mymethod", func, [ua.VariantType.Int64], [ua.VariantType.Boolean]
    #)
    #multiply_node = await myobj.add_method(
    #    idx,
    #    "multiply",
    #    multiply,
    #    [ua.VariantType.Int64, ua.VariantType.Int64],
    #    [ua.VariantType.Int64],
    #)

    # import some nodes from xml
    #await server.import_xml("custom_nodes.xml")

    # creating a default event object
    # The event object automatically will have members for all events properties
    # you probably want to create a custom event type, see other examples
    #myevgen = await server.get_event_generator()
    #myevgen.event.Severity = 300

    # starting!
    async with server:
        print("Available loggers are: ", logging.Logger.manager.loggerDict.keys())
        # enable following if you want to subscribe to nodes on server side
        # handler = SubHandler()
        # sub = await server.create_subscription(500, handler)
        # handle = await sub.subscribe_data_change(myvar)
        # trigger event, all subscribed clients wil receive it
        var = await array_var.read_value()  # return a ref to value in db server side! not a copy!
        var = copy.copy(
            var
        )  # WARNING: we need to copy before writting again otherwise no data change event will be generated
        var.append(9.3)
        await array_var.write_value(var)
        await device1.write_value("Running")
        #await myevgen.trigger(message="This is BaseEvent")
        # write_attribute_value is a server side method which is faster than using write_value
        # but than methods has less checks
        await server.write_attribute_value(server_var.nodeid, ua.DataValue(0.9))
        
        # running the server
        while True:
            await asyncio.sleep(0.1)
            await server.write_attribute_value(server_var.nodeid, ua.DataValue(sin(time.time())))


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    asyncio.run(main())